In [1]:
!pip install roboflow ultralytics sahi

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 49.9/49.9 MB 24.2 MB/s  0:00:02m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 20.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.2/4.2 MB 27.1 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 14.9 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 63.0/63.0 MB 27.0 MB/s  0:00:02m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 16.5 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.7/8.7 MB 31.5 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 14.7 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.6/6.6 MB 31.2 MB/s  0:00:00m0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 35.7/35.7 MB 42.8 MB/s  0:00:00m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 30.7 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 772.7/772.7 kB 14.4 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
from roboflow import Roboflow
rf = Roboflow(api_key="mz3cNkxiO8av9JAjZbS3")
project = rf.workspace("my-ws-lwkgs").project("tl_detector-coivv")
version = project.version(29)
dataset = version.download("coco")

loading Roboflow workspace...
loading Roboflow project...


Extracting Dataset Version Zip to tl_detector-29 in coco:: 100%|██████████| 1614/1614 [00:13<00:00, 119.70it/s]


In [3]:
from sahi.slicing import slice_coco
from sahi.utils.file import save_json
from sahi.utils.coco import Coco
import os

out_dir = f"{dataset.location}/sliced"
try:
    os.rmdir(out_dir)
except:
    pass

coco_dict, coco_path = slice_coco(
    coco_annotation_file_path=f"{dataset.location}/train/_annotations.coco.json",
    image_dir=f"{dataset.location}/train",
    output_coco_annotation_file_name="annotations",
    output_dir=f"{out_dir}/images",
    slice_height=512,
    slice_width=512,
    overlap_height_ratio=0.2,
    overlap_width_ratio=0.2,
    min_area_ratio=0.1,  # Add this
    ignore_negative_samples=False,
)

coco = Coco.from_coco_dict_or_path(coco_dict, image_dir=f"{out_dir}/images")
result = coco.split_coco_as_train_val(train_split_rate=0.85)

Loading coco annotations: 100%|██████████| 29115/29115 [00:00<00:00, 31773.00it/s]


In [4]:
from sahi.utils.coco import Coco, export_coco_as_yolo

sliced_yolo_dir=f"{out_dir}/yolo_dataset"
try:
    os.rmdir(sliced_yolo_dir)
except:
    pass

data_yml_path = export_coco_as_yolo(
    output_dir=sliced_yolo_dir,
    train_coco=result["train_coco"],
    val_coco=result["val_coco"]
)

2025-10-15 18:03:21,153 - sahi - INFO - generating image symlinks and annotation files for yolo... (coco.py:1576)
100%|██████████| 24747/24747 [01:18<00:00, 314.13it/s]
2025-10-15 18:04:39,935 - sahi - INFO - generating image symlinks and annotation files for yolo... (coco.py:1576)
100%|██████████| 4368/4368 [00:15<00:00, 279.22it/s]


In [5]:
from ultralytics import YOLO

#sliced_yolo_dir="/workspace/tl_detector-19/sliced/yolo_dataset"

model = YOLO('yolov8n.pt')
results = model.train(
    data=f"{sliced_yolo_dir}/data.yml",
    epochs=100,
    imgsz=320,
    rect=False,
    multi_scale=False,
    batch=128,
    workers=4,
    name='tl_detector',
    augment=True,  # включаем ручной контроль над аугментацией
    hsv_h=0.0,
    hsv_s=0.0,
    hsv_v=0.0,
    fliplr=0.0,
    flipud=0.0,
    mosaic=0.0,
    mixup=0.0,
    cutmix=0.0,
    copy_paste=0.0,
    auto_augment='none',
    erasing=0.0
)

WARNING ⚠️ user config directory '/root/.config/Ultralytics' is not writeable, using '/tmp/Ultralytics'. Set YOLO_CONFIG_DIR to override.
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/tmp/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.3.214 🚀 Python-3.12.3 torch-2.8.0+cu128 CUDA:0 (NVIDIA GeForce RTX 3090, 24134MiB)
engine/trainer: agnostic_nms=False, amp=True, augment=True, auto_augment=none, batch=128, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/workspace/tl_detector-29/sliced/yolo_dataset/data.yml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, epochs=100, erasing=0.0, exist_ok=F

In [6]:
print(f"Result: {results.save_dir}/confusion_matrix_normalized.png")
print(f"Model: {results.save_dir}/weights/best.pt")
print(f"Calibration images: {out_dir}/images")

Result: /workspace/runs/detect/tl_detector/confusion_matrix_normalized.png
Model: /workspace/runs/detect/tl_detector/weights/best.pt
Calibration images: /workspace/tl_detector-29/sliced/images


In [7]:
import time
import functools
import requests

TOKEN = "8212098701:AAHKTZpdO8FRoxQF9bJBHte_EupMDWWD3Ls"
CHAT_ID = "390672240"

def notify(text):
    requests.get(f"https://api.telegram.org/bot{TOKEN}/sendMessage", params={
        "chat_id": CHAT_ID,
        "text": text
    })

notify('Finished')